In [ ]:
import concurrent.futures

import matplotlib.pyplot as plt
import numpy as np
import torch
from hydra import compose, initialize
from hydra.utils import instantiate
from tqdm import tqdm

plt.rcParams["font.family"] = "serif"

import pytorch_lightning as pl
from pytorch_lightning.utilities import move_data_to_device

from bliss.global_env import GlobalEnv

device = torch.device("cuda:5" if torch.cuda.is_available() else "cpu")

Configure encoder to make predictions using weights from a previous training run:

In [ ]:
ckpt = "/data/scratch/descwl/checkpoints/trained_encoder_descwl_setting1.ckpt"

with initialize(config_path="../", version_base=None):
    cfg = compose(
        "config_train_npe",
        {
            "train.pretrained_weights=" + ckpt,
        },
    )

seed = pl.seed_everything(cfg.train.seed)
GlobalEnv.seed_in_this_program = seed

Configure the test dataloader:

In [ ]:
data_source = instantiate(cfg.train.data_source)
data_source.setup("test")

Load in encoder weights:

In [ ]:
encoder = instantiate(cfg.encoder).to(device)
encoder_state_dict = torch.load(cfg.train.pretrained_weights, map_location=device)["state_dict"]
encoder.load_state_dict(encoder_state_dict)
encoder = encoder.eval()

In [ ]:
confidence_levels = torch.linspace(0.05, 0.95, steps=19)

ci_quantiles = torch.distributions.Normal(0, 1).icdf(1 - (1 - confidence_levels) / 2).to(device)

In [ ]:
# Load precomputed results from script
# Run: python compute_npe_credibleintervals.py (configure in config_compute_npe_credibleintervals.yaml)
cached = torch.load("npe_credible_intervals_setting1.pt", weights_only=False)

confidence_levels = cached["confidence_levels"]
shear1_true = cached["shear1_true"]
shear2_true = cached["shear2_true"]
shear1_ci_lower = cached["shear1_ci_lower"]
shear1_ci_upper = cached["shear1_ci_upper"]
shear2_ci_lower = cached["shear2_ci_lower"]
shear2_ci_upper = cached["shear2_ci_upper"]

print(f"Loaded {len(shear1_true)} samples")

In [ ]:
shear1_coverage_probs = (
    (
        (shear1_ci_lower <= shear1_true.unsqueeze(-1))
        * (shear1_true.unsqueeze(-1) <= shear1_ci_upper)
    )
    .float()
    .mean(0)
)
shear2_coverage_probs = (
    (
        (shear2_ci_lower <= shear2_true.unsqueeze(-1))
        * (shear2_true.unsqueeze(-1) <= shear2_ci_upper)
    )
    .float()
    .mean(0)
)

In [ ]:
for i, ci in enumerate(confidence_levels):
    print(
        f"Confidence level: {ci:.2f}, Shear 1: {shear1_coverage_probs[i]:.4f}, Shear 2: {shear2_coverage_probs[i]:.4f}"
    )

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 6))
fontsize = 24
ticklabelsize = 16
color = "#741d91"
s = 180

_ = ax[0].axline((0, 0), slope=1, linestyle="dotted", color="black", linewidth=2, zorder=0)
_ = ax[0].scatter(confidence_levels, shear1_coverage_probs, color=color, s=s, zorder=1)
_ = ax[0].set_title(r"$\gamma_1$", fontsize=1.5 * fontsize)
_ = ax[0].set_xlabel("Nominal coverage probability", fontsize=fontsize)
_ = ax[0].set_ylabel("Empirical coverage probability", fontsize=fontsize)
_ = ax[0].tick_params(axis="both", which="major", labelsize=ticklabelsize)
_ = ax[0].set_xlim(0, 1)
_ = ax[0].set_ylim(0, 1)

_ = ax[1].axline((0, 0), slope=1, linestyle="dotted", color="black", linewidth=2, zorder=0)
_ = ax[1].scatter(confidence_levels, shear2_coverage_probs, color=color, s=s, zorder=1)
_ = ax[1].set_title(r"$\gamma_2$", fontsize=1.5 * fontsize)
_ = ax[1].set_xlabel("Nominal coverage probability", fontsize=fontsize)
_ = ax[1].set_ylabel("Empirical coverage probability", fontsize=fontsize)
_ = ax[1].tick_params(axis="both", which="major", labelsize=ticklabelsize)
_ = ax[1].set_xlim(0, 1)
_ = ax[1].set_ylim(0, 1)

for a in ax.flat:
    _ = a.spines[["top", "right"]].set_visible(False)

fig.tight_layout()

fig.savefig(
    "figures/descwl_coverageprobs.png",
    dpi=300,
    transparent=True,
    bbox_inches="tight",
    pad_inches=0,
)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 6))
fontsize = 32
ticklabelsize = 16
color = np.array(["#9c7c3d", "#741d91"])
alpha = 0.5

axmin = -0.06
axmax = 0.06

np.random.seed(0)
n_samples = len(shear1_true)  # Use actual sample count, not batch count
indexes = np.arange(n_samples)

interval_idx = 17  # 90% credible interval

_ = ax[0].axline((0, 0), slope=1, linestyle="dotted", color="black", linewidth=2)
shear1_coverage = (
    (shear1_ci_lower <= shear1_true.unsqueeze(-1)) * (shear1_true.unsqueeze(-1) <= shear1_ci_upper)
)[..., interval_idx]

covered_legend = False
uncovered_legend = False

for i in range(n_samples):
    if (shear1_coverage[indexes[i]]) and (not covered_legend):
        covered_legend = True
        _ = ax[0].vlines(
            x=shear1_true[indexes[i]],
            ymin=shear1_ci_lower[..., interval_idx][indexes[i]],
            ymax=shear1_ci_upper[..., interval_idx][indexes[i]],
            alpha=alpha,
            color=color[shear1_coverage[indexes[i]]],
            label="covers",
        )
    elif (not shear1_coverage[indexes[i]]) and (not uncovered_legend):
        uncovered_legend = True
        _ = ax[0].vlines(
            x=shear1_true[indexes[i]],
            ymin=shear1_ci_lower[..., interval_idx][indexes[i]],
            ymax=shear1_ci_upper[..., interval_idx][indexes[i]],
            alpha=alpha,
            color=color[shear1_coverage[indexes[i]]],
            label="does not cover",
        )
    else:
        _ = ax[0].vlines(
            x=shear1_true[indexes[i]],
            ymin=shear1_ci_lower[..., interval_idx][indexes[i]],
            ymax=shear1_ci_upper[..., interval_idx][indexes[i]],
            alpha=alpha,
            color=color[shear1_coverage[indexes[i]]],
        )
_ = ax[0].set_xlabel(r"$\gamma_1$", fontsize=fontsize)
_ = ax[0].set_ylabel(r"Posterior $\gamma_1$", fontsize=fontsize)
_ = ax[0].tick_params(axis="both", which="major", labelsize=ticklabelsize)
_ = ax[0].legend(loc="upper left", prop={"size": ticklabelsize})
_ = ax[0].set_xlim(axmin, axmax)
_ = ax[0].set_ylim(axmin, axmax)


_ = ax[1].axline((0, 0), slope=1, linestyle="dotted", color="black", linewidth=2)
shear2_coverage = (
    (shear2_ci_lower <= shear2_true.unsqueeze(-1)) * (shear2_true.unsqueeze(-1) <= shear2_ci_upper)
)[..., interval_idx]

covered_legend = False
uncovered_legend = False

for i in range(n_samples):
    if (shear2_coverage[indexes[i]]) and (not covered_legend):
        covered_legend = True
        _ = ax[1].vlines(
            x=shear2_true[indexes[i]],
            ymin=shear2_ci_lower[..., interval_idx][indexes[i]],
            ymax=shear2_ci_upper[..., interval_idx][indexes[i]],
            alpha=alpha,
            color=color[shear2_coverage[indexes[i]]],
            label="covers",
        )
    elif (not shear2_coverage[indexes[i]]) and (not uncovered_legend):
        uncovered_legend = True
        _ = ax[1].vlines(
            x=shear2_true[indexes[i]],
            ymin=shear2_ci_lower[..., interval_idx][indexes[i]],
            ymax=shear2_ci_upper[..., interval_idx][indexes[i]],
            alpha=alpha,
            color=color[shear2_coverage[indexes[i]]],
            label="does not cover",
        )
    else:
        _ = ax[1].vlines(
            x=shear2_true[indexes[i]],
            ymin=shear2_ci_lower[..., interval_idx][indexes[i]],
            ymax=shear2_ci_upper[..., interval_idx][indexes[i]],
            alpha=alpha,
            color=color[shear2_coverage[indexes[i]]],
        )
_ = ax[1].set_xlabel(r"$\gamma_2$", fontsize=fontsize)
_ = ax[1].set_ylabel(r"Posterior $\gamma_2$", fontsize=fontsize)
_ = ax[1].tick_params(axis="both", which="major", labelsize=ticklabelsize)
_ = ax[1].set_xlim(axmin, axmax)
_ = ax[1].set_ylim(axmin, axmax)

leg = ax[0].legend(loc="upper left", prop={"size": ticklabelsize})
for lh in leg.legend_handles:
    lh.set_alpha(1)
for a in ax.flat:
    _ = a.spines[["top", "right"]].set_visible(False)

_ = fig.tight_layout()

fig.savefig(
    "figures/descwl_credibleintervals.png",
    dpi=300,
    transparent=True,
    bbox_inches="tight",
    pad_inches=0,
)